In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [2]:
train = pd.read_csv('../data/labels/train_sent_emo.csv')
manifest = pd.read_csv('../data/manifest.csv')

In [3]:
X_text = np.load("../cache/text.npy")
X_audio = np.load("../cache/audio_wavlm_13L.npy").astype(np.float32).reshape(-1, 13, 1536)

In [4]:

tr = (manifest.split == "train").values
va = (manifest.split == "dev").values
te = (manifest.split == "test").values

Xt_train, Xa_train = torch.from_numpy(X_text[tr]), torch.from_numpy(X_audio[tr])
Xt_val,   Xa_val   = torch.from_numpy(X_text[va]), torch.from_numpy(X_audio[va])

In [5]:
EMOTIONS = {"neutral":0, "joy":1, "surprise":2, "anger":3, "sadness":4, "disgust":5, "fear":6}
manifest['emo_encoded'] = manifest["Emotion"].map(EMOTIONS)
Y_train = manifest[manifest.split == 'train'].emo_encoded.to_numpy()
Y_train = torch.from_numpy(Y_train)

Y_val = manifest[manifest.split == 'dev'].emo_encoded.to_numpy()
Y_val = torch.from_numpy(Y_val)

Y_test = manifest[manifest.split == 'test'].emo_encoded.to_numpy()
Y_test = torch.from_numpy(Y_test)

In [6]:
class EmotionClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer_w = nn.Parameter(torch.zeros(13))
        self.network = nn.Sequential(
            nn.Linear(2304, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(128, len(EMOTIONS))
        )

    def forward(self, x_text, x_audio):
        w = torch.softmax(self.layer_w, 0)
        pooled = (x_audio * w[None, :, None]).sum(1)
        return self.network(torch.cat([x_text, pooled], 1))


counts = torch.bincount(Y_train)
class_weights = (1.0 / counts.float().sqrt())
class_weights = class_weights / class_weights.sum() * len(EMOTIONS)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)




In [7]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score


model = EmotionClassifier()

optimizer = torch.optim.AdamW([
    {"params": model.network.parameters(), "lr": 1e-4, "weight_decay": 1e-2},
    {"params": [model.layer_w], "lr": 1e-2, "weight_decay": 0.0},
])

#train_ds = TensorDataset(X_train, Y_train)
train_dl = DataLoader(TensorDataset(Xt_train, Xa_train, Y_train), batch_size=256, shuffle=True)

best_f1 = -1
patience = 50
patience_counter = 0
best_weights = {k: v.clone() for k, v in model.state_dict().items()}


best_val_loss, best_weights, stale = float("inf"), None, 0
patience = 10

for epoch in range(100):
    model.train()
    for xt, xa, yb in train_dl:
        logits = model(xt, xa)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xt_val, Xa_val), Y_val).item()
        val_f1 = f1_score(Y_val, model(Xt_val, Xa_val).argmax(1), average="weighted")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        stale = 0
        flag=''
    else:
        stale += 1
        flag = "*"
    print(f"epoch {epoch:3d}  val_loss {val_loss:.4f}  dev_wF1 {val_f1:.4f}{flag}")
    if stale >= patience:
        break

model.load_state_dict(best_weights)

epoch   0  val_loss 1.9144  dev_wF1 0.0815
epoch   1  val_loss 1.6568  dev_wF1 0.4860
epoch   2  val_loss 1.5818  dev_wF1 0.5096
epoch   3  val_loss 1.5474  dev_wF1 0.5421
epoch   4  val_loss 1.5117  dev_wF1 0.5456
epoch   5  val_loss 1.5080  dev_wF1 0.5627
epoch   6  val_loss 1.4738  dev_wF1 0.5524*
epoch   7  val_loss 1.4524  dev_wF1 0.5600*
epoch   8  val_loss 1.4469  dev_wF1 0.5598*
epoch   9  val_loss 1.4294  dev_wF1 0.5669
epoch  10  val_loss 1.4324  dev_wF1 0.5652*
epoch  11  val_loss 1.4088  dev_wF1 0.5710
epoch  12  val_loss 1.4206  dev_wF1 0.5513*
epoch  13  val_loss 1.3788  dev_wF1 0.5860
epoch  14  val_loss 1.3706  dev_wF1 0.5856*
epoch  15  val_loss 1.3662  dev_wF1 0.5970
epoch  16  val_loss 1.3820  dev_wF1 0.5894*
epoch  17  val_loss 1.3581  dev_wF1 0.5899*
epoch  18  val_loss 1.3573  dev_wF1 0.6074
epoch  19  val_loss 1.3637  dev_wF1 0.5952*
epoch  20  val_loss 1.3357  dev_wF1 0.5904*
epoch  21  val_loss 1.3649  dev_wF1 0.5948*
epoch  22  val_loss 1.4024  dev_wF1 0.5784*

<All keys matched successfully>

In [8]:

model.eval()
with torch.no_grad():
    tr_pred = model(Xt_train, Xa_train).argmax(1)
    va_pred = model(Xt_val, Xa_val).argmax(1)

print("train wF1", f1_score(Y_train, tr_pred, average="weighted"))
print("dev   wF1", f1_score(Y_val, va_pred, average="weighted"))

train wF1 0.7059692129832029
dev   wF1 0.6086417074733165


In [9]:
from sklearn.metrics import classification_report
print(classification_report(Y_val, va_pred, target_names=EMOTIONS, zero_division=0))

              precision    recall  f1-score   support

     neutral       0.75      0.73      0.74       469
         joy       0.55      0.60      0.58       163
    surprise       0.56      0.73      0.63       150
       anger       0.55      0.48      0.52       153
     sadness       0.53      0.37      0.43       111
     disgust       0.10      0.18      0.12        22
        fear       0.30      0.20      0.24        40

    accuracy                           0.61      1108
   macro avg       0.48      0.47      0.47      1108
weighted avg       0.62      0.61      0.61      1108



In [10]:
print(torch.softmax(model.layer_w, 0).round(decimals=3))

tensor([0.1460, 0.0790, 0.0790, 0.0930, 0.1120, 0.0160, 0.0290, 0.0630, 0.1500,
        0.1120, 0.0510, 0.0430, 0.0280], grad_fn=<RoundBackward1>)


In [11]:
torch.save(model.state_dict(), "../cache/fusion_model.pt")